نحط هالكود أولا بنهاية النوت بوك.

In [ ]:
# ==========================================================
# Enhanced evaluation + automatic saving utilities (FINAL)
# ==========================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from textwrap import wrap


def _ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
    return path


def save_confusion_matrix_figure(cm, classes, outpath, title="Confusion Matrix", cmap=None):
    fig, ax = plt.subplots(figsize=(6, 5))
    if cmap is None:
        cmap = "Blues"
    sns.heatmap(cm, annot=True, fmt=".2f", ax=ax, cmap=cmap, cbar=True)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def evaluate_evasion_performance(
    y_true,
    y_pred_clean,
    y_pred_adv,
    y_score_clean=None,
    y_score_adv=None,
    normal_class_label=0,
    model_name="Model",
    attack_name="Attack",
    save=True,
    save_dir="evaluation_results",
    prefix=None,
    classes=None
):
    """
    Enhanced unified evaluation for IDS under evasion attacks, with optional automatic saving.
    Returns:
      df_metrics (pd.DataFrame) rounded to 4 decimals,
      save_info (dict) with paths to saved files if save=True (otherwise None)
    """
    sns.set_theme(style="whitegrid", context="notebook")
    plt.rcParams["figure.dpi"] = 140

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if prefix is None:
        folder_name = f"{timestamp}__{model_name}__{attack_name}".replace(" ", "_").replace("/", "-")
    else:
        folder_name = f"{timestamp}__{prefix}".replace(" ", "_").replace("/", "-")

    out_base = os.path.join(save_dir, folder_name)
    if save:
        _ensure_dir(out_base)

    print(f"\n{'='*12} {model_name} — {attack_name} Evaluation {'='*12}\n")

    # ===== Metadata Overview =====
    print("📋 Metadata Overview:")
    print(f" - Timestamp: {timestamp}")
    print(f" - Model: {model_name}")
    print(f" - Attack: {attack_name}")
    print(f" - Total Samples: {len(y_true)}")
    print(f" - Save Directory: {out_base if save else 'Saving Disabled'}")
    print("="*55 + "\n")

    # ===== 1. Compute metrics =====
    metrics = {}
    for name, y_pred in {"Clean": y_pred_clean, "Adversarial": y_pred_adv}.items():
        metrics[name] = {
            "Accuracy": float(accuracy_score(y_true, y_pred)),
            "Precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
            "Recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
            "F1-score": float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
        }

    asr = float(np.mean(np.array(y_pred_clean) != np.array(y_pred_adv)))
    metrics["Adversarial"]["ASR"] = asr

    # Optional AUC (for Normal class)
    if y_score_clean is not None and y_score_adv is not None:
        try:
            metrics["Clean"]["AUC"] = float(
                roc_auc_score(
                    (np.array(y_true) == normal_class_label).astype(int),
                    np.array(y_score_clean)[:, normal_class_label]
                )
            )
            metrics["Adversarial"]["AUC"] = float(
                roc_auc_score(
                    (np.array(y_true) == normal_class_label).astype(int),
                    np.array(y_score_adv)[:, normal_class_label]
                )
            )
        except Exception as e:
            print("AUC calculation skipped:", e)

    # ===== 2. Tabular summary =====
    df_metrics = pd.DataFrame(metrics).T
    display(df_metrics.round(4))

    # ===== 3. Classification report (Adversarial only) =====
    clf_report_dict = classification_report(y_true, y_pred_adv, output_dict=True, zero_division=0)
    clf_report_str = classification_report(y_true, y_pred_adv, digits=4, zero_division=0)
    print("\nDetailed Classification Report (Adversarial Predictions):\n")
    print(clf_report_str)

    # ===== 4. Confusion matrices =====
    cm_clean = confusion_matrix(y_true, y_pred_clean, normalize="true")
    cm_adv = confusion_matrix(y_true, y_pred_adv, normalize="true")

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(cm_clean, cmap="Blues", ax=ax[0], annot=False)
    ax[0].set_title("Normalized Confusion Matrix — Clean")
    ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")

    sns.heatmap(cm_adv, cmap="Reds", ax=ax[1], annot=False)
    ax[1].set_title("Normalized Confusion Matrix — Adversarial")
    ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")

    plt.suptitle(f"{model_name} — {attack_name}: Class Confusion Comparison", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # ===== 5. Metric comparison plots =====
    df_compare = df_metrics[["Accuracy", "Precision", "Recall", "F1-score"]].T
    df_compare["Δ (drop)"] = df_compare["Clean"] - df_compare["Adversarial"]

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    df_compare[["Clean", "Adversarial"]].plot(kind="bar", ax=ax[0])
    ax[0].set_title("Metric Comparison Before vs After Attack")
    ax[0].set_ylabel("Score")
    ax[0].legend(loc="lower right")

    sns.barplot(x=df_compare.index, y=df_compare["Δ (drop)"], ax=ax[1])
    ax[1].set_title("Performance Drop per Metric")
    ax[1].set_ylabel("Δ = Clean - Adversarial")
    ax[1].axhline(0, color="gray", linestyle="--")

    plt.suptitle(f"{model_name} — {attack_name}: Performance Impact Overview", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

    # ===== 6. ASR visualization =====
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(["Attack Success Rate"], [asr * 100])
    ax.set_title("Evasion Success (ASR)")
    ax.set_ylim(0, 100)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

    # ===== 7. Text summary =====
    print("\nSummary Insights:")
    print(f" - Clean Accuracy: {metrics['Clean']['Accuracy']:.4f}")
    print(f" - Adversarial Accuracy: {metrics['Adversarial']['Accuracy']:.4f}")
    print(f" - Attack Success Rate (ASR): {asr*100:.2f}%")
    print(f" - Drop in Accuracy: {(metrics['Clean']['Accuracy']-metrics['Adversarial']['Accuracy'])*100:.2f}%")
    print(f" - Drop in F1-score: {(metrics['Clean']['F1-score']-metrics['Adversarial']['F1-score'])*100:.2f}%")
    if "AUC" in metrics["Clean"]:
        delta_auc = metrics["Clean"]["AUC"] - metrics["Adversarial"]["AUC"]
        print(f" - AUC Change (Normal class): {delta_auc:+.4f}")
    print(f"\nEvaluation completed for {model_name} under {attack_name}.\n")

    # ===== 8. Saving outputs =====
    save_info = None
    if save:
        save_info = {}
        metrics_excel = os.path.join(out_base, "metrics_summary.xlsx")
        metrics_csv = os.path.join(out_base, "metrics_summary.csv")
        df_metrics.round(6).to_excel(metrics_excel)
        df_metrics.round(6).to_csv(metrics_csv, index=True)
        save_info["metrics_excel"] = metrics_excel
        save_info["metrics_csv"] = metrics_csv

        clf_json = os.path.join(out_base, "classification_report_adversarial.json")
        clf_txt = os.path.join(out_base, "classification_report_adversarial.txt")
        with open(clf_json, "w", encoding="utf-8") as f:
            json.dump(clf_report_dict, f, indent=2)
        with open(clf_txt, "w", encoding="utf-8") as f:
            f.write(clf_report_str)
        save_info["classification_report_json"] = clf_json
        save_info["classification_report_txt"] = clf_txt

        cm_clean_npy = os.path.join(out_base, "cm_clean.npy")
        cm_adv_npy = os.path.join(out_base, "cm_adv.npy")
        np.save(cm_clean_npy, cm_clean)
        np.save(cm_adv_npy, cm_adv)
        save_info["cm_clean_npy"] = cm_clean_npy
        save_info["cm_adv_npy"] = cm_adv_npy

        classes_arg = classes if classes is not None else [str(i) for i in range(cm_clean.shape[0])]
        cm_clean_png = os.path.join(out_base, "cm_clean.png")
        cm_adv_png = os.path.join(out_base, "cm_adv.png")
        save_confusion_matrix_figure(cm_clean, classes_arg, cm_clean_png, title="Normalized Confusion Matrix - Clean", cmap="Blues")
        save_confusion_matrix_figure(cm_adv, classes_arg, cm_adv_png, title="Normalized Confusion Matrix - Adversarial", cmap="Reds")
        save_info["cm_clean_png"] = cm_clean_png
        save_info["cm_adv_png"] = cm_adv_png

        preds_npz = os.path.join(out_base, "predictions_and_labels.npz")
        np.savez_compressed(preds_npz,
                            y_true=np.array(y_true),
                            y_pred_clean=np.array(y_pred_clean),
                            y_pred_adv=np.array(y_pred_adv))
        save_info["preds_npz"] = preds_npz

        meta = {
            "model_name": model_name,
            "attack_name": attack_name,
            "timestamp": timestamp,
            "rows": int(len(y_true)),
            "asr": asr,
            "paths": save_info
        }
        meta_path = os.path.join(out_base, "metadata.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)
        save_info["metadata"] = meta_path

        print(f"Saved evaluation outputs to: {out_base}")

    return df_metrics.round(4), save_info

بعده نستدعيه من خلال وضع هالكود بخلية آخرى بعد تشغيل الهجمات


In [ ]:
# === Empty-template invocation for evaluate_evasion_performance ===
# Fill the placeholders below with your actual arrays / predictions / names before running.

df_eval, save_info = evaluate_evasion_performance(
    y_true=None,             # <- replace with your true labels (list / np.array)
    y_pred_clean=None,       # <- replace with clean predictions (model.predict(X_test))
    y_pred_adv=None,         # <- replace with adversarial predictions (same length as y_true)
    y_score_clean=None,      # <- optional: probability scores (np.array shape [n_samples, n_classes])
    y_score_adv=None,        # <- optional: same shape as y_score_clean for adversarial scores
    normal_class_label=0,    # <- label index/value considered "normal"/benign (adjust if needed)
    model_name="",           # <- e.g. "Random_Forest_v1" (string)
    attack_name="",          # <- e.g. "FGSM_eps0.02" (string)
    save=False,              # <- set True to save outputs (Excel/JSON/PNG) to disk
    save_dir="",             # <- base folder for saving when save=True, e.g. "eval_outputs"
    prefix=None,             # <- optional custom folder prefix (string) or None
    classes=None             # <- optional list of class labels for plots, e.g. ['Benign','Attack']
)
# --- Output summary ---
print("\n✅ Evaluation completed successfully!")
print("📊 Metrics DataFrame:")
display(df_eval)

print("\n📁 Saved files summary:")
for k, v in save_info.items():
    print(f" - {k}: {v}")

| العنصر                           | الوصف                                                               |
| -------------------------------- | ------------------------------------------------------------------- |
| 🔹 **جدول تفاعلي**               | يعرض كل المقاييس (Accuracy, Precision, Recall, F1, ASR, AUC) بوضوح. |
| 🔹 **مقارنة بصرية**              | رسوم بيانية توضح الفرق بين الأداء قبل وبعد الهجوم.                  |
| 🔹 **Δ drop plot**               | يوضّح مقدار الانخفاض في كل مقياس (بشكل أفقي وبالألوان).             |
| 🔹 **ASR bar**                   | يبيّن نسبة نجاح الهجوم باللون الأحمر.                               |
| 🔹 **تقرير تفصيلي للفئات**       | يوضح نقاط الضعف في الفئات الصغيرة بعد الهجوم.                       |
| 🔹 **مصفوفات الالتباس المزدوجة** | Heatmaps توضح كيف تغيّر التنبؤ بين الحالتين.                        |
